In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run /Workspace/Users/syedmatin2511@gmail.com/consolidated_pipeline/01_Setup/utilities

In [0]:
print(bronze_schema, silver_schema, gold_schema)

bronze silver gold


In [0]:
dbutils.widgets.text("catalog", "fmcg", "Catalog")
dbutils.widgets.text("data_source", "orders", "Data Source")

catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

base_path = f's3://sportsbar-jam/{data_source}'
landing_path = f"{base_path}/landing/"
processed_path = f"{base_path}/processed/"
print("Base Path: ", base_path)
print("Landing Path: ", landing_path)
print("Processed Path: ", processed_path)


# define the tables
bronze_table = f"{catalog}.{bronze_schema}.{data_source}"
silver_table = f"{catalog}.{silver_schema}.{data_source}"
gold_table = f"{catalog}.{gold_schema}.sb_fact_{data_source}"

Base Path:  s3://sportsbar-jam/orders
Landing Path:  s3://sportsbar-jam/orders/landing/
Processed Path:  s3://sportsbar-jam/orders/processed/


In [0]:
df = spark.read.options(header=True, inferSchema=True).csv(f"{landing_path}/*.csv").withColumn("read_timestamp", F.current_timestamp()).select("*", "_metadata.file_name", "_metadata.file_size")

print("Total Rows: ", df.count())
df.show(5)

Total Rows:  51810
+------------+--------------------+-----------+----------+---------+--------------------+--------------------+---------+
|    order_id|order_placement_date|customer_id|product_id|order_qty|      read_timestamp|           file_name|file_size|
+------------+--------------------+-----------+----------+---------+--------------------+--------------------+---------+
|FOCT62720602|Tuesday, Septembe...|     ABC987|  25891301|     71.0|2026-03-16 10:57:...|orders_2025_09_30...|    41446|
|FOCT62720602|Tuesday, Septembe...|     789720|  25891502|    125.0|2026-03-16 10:57:...|orders_2025_09_30...|    41446|
|FOCT62720602|Tuesday, Septembe...|     789720|  25891403|    462.0|2026-03-16 10:57:...|orders_2025_09_30...|    41446|
|FOCT62720602|Tuesday, Septembe...|    INVALID|  25891601|    133.0|2026-03-16 10:57:...|orders_2025_09_30...|    41446|
|FOCT62720602|Tuesday, Septembe...|     789720|  25891602|     79.0|2026-03-16 10:57:...|orders_2025_09_30...|    41446|
+------------

In [0]:
df.write\
 .format("delta") \
 .option("delta.enableChangeDataFeed", "true") \
 .mode("append") \
 .saveAsTable(bronze_table)

In [0]:
files = dbutils.fs.ls(landing_path)
for file_info in files:
    dbutils.fs.mv(
        file_info.path,
        f"{processed_path}/{file_info.name}",
        True
    )

In [0]:
df_orders = spark.sql(f"SELECT * FROM {bronze_table}")
df_orders.show(2)

+------------+--------------------+-----------+----------+---------+--------------------+--------------------+---------+
|    order_id|order_placement_date|customer_id|product_id|order_qty|      read_timestamp|           file_name|file_size|
+------------+--------------------+-----------+----------+---------+--------------------+--------------------+---------+
|FJUL33320501|          2025/07/01|     789320|  25891203|    150.0|2026-03-16 10:58:...|orders_2025_07_01...|    20744|
|FJUL33320501|          2025/07/01|     789320|  25891301|     46.0|2026-03-16 10:58:...|orders_2025_07_01...|    20744|
+------------+--------------------+-----------+----------+---------+--------------------+--------------------+---------+
only showing top 2 rows


In [0]:
# 1. Keep only rows where order_qty is present
df_orders = df_orders.filter(F.col("order_qty").isNotNull())


# 2. Clean customer_id → keep numeric, else set to 999999
df_orders = df_orders.withColumn(
    "customer_id",
    F.when(F.col("customer_id").rlike("^[0-9]+$"), F.col("customer_id"))
     .otherwise("999999")
     .cast("string")
)

# 3. Remove weekday name from the date text
#    "Tuesday, July 01, 2025" → "July 01, 2025"
df_orders = df_orders.withColumn(
    "order_placement_date",
    F.regexp_replace(F.col("order_placement_date"), r"^[A-Za-z]+,\s*", "")
)

# 4. Parse order_placement_date using multiple possible formats
df_orders = df_orders.withColumn(
    "order_placement_date",
    F.coalesce(
        F.try_to_date("order_placement_date", "yyyy/MM/dd"),
        F.try_to_date("order_placement_date", "dd-MM-yyyy"),
        F.try_to_date("order_placement_date", "dd/MM/yyyy"),
        F.try_to_date("order_placement_date", "MMMM dd, yyyy"),
    )
)

# 5. Drop duplicates
df_orders = df_orders.dropDuplicates(["order_id", "order_placement_date", "customer_id", "product_id", "order_qty"])

# 5. convert product id to string
df_orders = df_orders.withColumn('product_id', F.col('product_id').cast('string'))

In [0]:
display(df_orders.limit(10))

order_id,order_placement_date,customer_id,product_id,order_qty,read_timestamp,file_name,file_size
FJUL33320501,2025-07-01,789320,25891203,150.0,2026-03-16T10:58:41.263Z,orders_2025_07_01.csv,20744
FJUL33320501,2025-07-01,789320,25891301,46.0,2026-03-16T10:58:41.263Z,orders_2025_07_01.csv,20744
FJUL33320501,2025-07-01,789320,25891201,354.0,2026-03-16T10:58:41.263Z,orders_2025_07_01.csv,20744
FJUL33320501,2025-07-01,789320,25891501,249.0,2026-03-16T10:58:41.263Z,orders_2025_07_01.csv,20744
FJUL33401603,2025-07-01,789401,25891302,40.0,2026-03-16T10:58:41.263Z,orders_2025_07_01.csv,20744
FJUL33401603,2025-07-01,789401,25891502,133.0,2026-03-16T10:58:41.263Z,orders_2025_07_01.csv,20744
FJUL33401603,2025-07-01,789401,25891503,145.0,2026-03-16T10:58:41.263Z,orders_2025_07_01.csv,20744
FJUL33401603,2025-07-01,789401,25891203,429.0,2026-03-16T10:58:41.263Z,orders_2025_07_01.csv,20744
FJUL33401603,2025-07-01,789401,25891201,461.0,2026-03-16T10:58:41.263Z,orders_2025_07_01.csv,20744
FJUL32101601,2025-07-01,789101,25891503,183.0,2026-03-16T10:58:41.263Z,orders_2025_07_01.csv,20744


In [0]:
df_products = spark.table("fmcg.silver.products")
df_joined = df_orders.join(df_products, on="product_id", how="inner").select(df_orders["*"], df_products["product_code"])

display(df_joined.limit(5))

order_id,order_placement_date,customer_id,product_id,order_qty,read_timestamp,file_name,file_size,product_code
FJUL32202603,2025-07-01,789202,25891103,370.0,2026-03-16T10:58:41.263Z,orders_2025_07_01.csv,20744,102628255d24304d6bbe0438b1ac992054f262e0814d306d0a34d7356cef3268
FJUL33321602,2025-07-01,789321,25891602,74.0,2026-03-16T10:58:41.263Z,orders_2025_07_01.csv,20744,778c2a7aa27bfdb211fd5ece048de80d00fbf3d6924bd908d91054796ba16ab6
FJUL34303602,2025-07-01,789303,25891102,317.0,2026-03-16T10:58:41.263Z,orders_2025_07_01.csv,20744,e92c739a8d78cd6cbe954648c2f9dd75ed61fcfd99b03e10dca65c3082d0728e
FJUL34622602,2025-07-01,789622,25891402,439.0,2026-03-16T10:58:41.263Z,orders_2025_07_01.csv,20744,fe5a8036be4b9a787b7c0ae013fc752a8cfb6c55a2f7b2fd152a6380925e9c49
FJUL33402303,2025-07-01,789402,25891303,63.0,2026-03-16T10:58:41.263Z,orders_2025_07_01.csv,20744,c68834ceaff15846bc1892c2185dc4e4f471d64fe3796b1a8ecc39a5a48c614f


In [0]:
if not (spark.catalog.tableExists(silver_table)):
    df_joined.write.format("delta").option(
        "delta.enableChangeDataFeed", "true"
    ).option("mergeSchema", "true").mode("overwrite").saveAsTable(silver_table)
else:
    silver_delta = DeltaTable.forName(spark, silver_table)
    silver_delta.alias("silver").merge(df_joined.alias("bronze"), "silver.order_placement_date = bronze.order_placement_date AND silver.order_id = bronze.order_id AND silver.product_code = bronze.product_code AND silver.customer_id = bronze.customer_id").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

### Gold

In [0]:
df_gold = spark.sql(f"SELECT order_id, order_placement_date as date, customer_id as customer_code, product_code, product_id, order_qty as sold_quantity FROM {silver_table};")

df_gold.show(2)

+------------+----------+-------------+--------------------+----------+-------------+
|    order_id|      date|customer_code|        product_code|product_id|sold_quantity|
+------------+----------+-------------+--------------------+----------+-------------+
|FJUL32202603|2025-07-01|       789202|102628255d24304d6...|  25891103|        370.0|
|FJUL33321602|2025-07-01|       789321|778c2a7aa27bfdb21...|  25891602|         74.0|
+------------+----------+-------------+--------------------+----------+-------------+
only showing top 2 rows


In [0]:
if not (spark.catalog.tableExists(gold_table)):
    print("creating New Table")
    df_gold.write.format("delta").option(
        "delta.enableChangeDataFeed", "true"
    ).option("mergeSchema", "true").mode("overwrite").saveAsTable(gold_table)
else:
    gold_delta = DeltaTable.forName(spark, gold_table)
    gold_delta.alias("source").merge(df_gold.alias("gold"), "source.date = gold.date AND source.order_id = gold.order_id AND source.product_code = gold.product_code AND source.customer_code = gold.customer_code").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

creating New Table


### Merge with Parent Company

In [0]:
df_child = spark.sql(f"SELECT date, product_code, customer_code, sold_quantity FROM {gold_table}")
df_child.show(10)

+----------+--------------------+-------------+-------------+
|      date|        product_code|customer_code|sold_quantity|
+----------+--------------------+-------------+-------------+
|2025-07-01|102628255d24304d6...|       789202|        370.0|
|2025-07-01|778c2a7aa27bfdb21...|       789321|         74.0|
|2025-07-01|e92c739a8d78cd6cb...|       789303|        317.0|
|2025-07-01|fe5a8036be4b9a787...|       789622|        439.0|
|2025-07-01|c68834ceaff15846b...|       789402|         63.0|
|2025-07-01|778c2a7aa27bfdb21...|       999999|        148.0|
|2025-07-01|778c2a7aa27bfdb21...|       789420|         76.0|
|2025-07-01|778c2a7aa27bfdb21...|       789201|         52.0|
|2025-07-01|e91ba9d665f90254d...|       789403|        379.0|
|2025-07-01|889c67757ece9c973...|       789421|         99.0|
+----------+--------------------+-------------+-------------+
only showing top 10 rows


In [0]:
df_monthly = (
    df_child
    # 1. Get month start date (e.g., 2025-11-30 → 2025-11-01)
    .withColumn("month_start", F.trunc("date", "MM"))   # or F.date_trunc("month", "date").cast("date")

    # 2.Group at monthly grain by month_start + product_code + customer_code
    .groupBy("month_start", "product_code", "customer_code")
    .agg(
        F.sum("sold_quantity").alias("sold_quantity")
    )

    # 3. Rename month_start back to `date` to match your target schema
    .withColumnRenamed("month_start", "date")
)

df_monthly.show(5, truncate=False)

+----------+----------------------------------------------------------------+-------------+-------------+
|date      |product_code                                                    |customer_code|sold_quantity|
+----------+----------------------------------------------------------------+-------------+-------------+
|2025-07-01|102628255d24304d6bbe0438b1ac992054f262e0814d306d0a34d7356cef3268|789202       |2944.0       |
|2025-07-01|778c2a7aa27bfdb211fd5ece048de80d00fbf3d6924bd908d91054796ba16ab6|789321       |1742.0       |
|2025-07-01|e92c739a8d78cd6cbe954648c2f9dd75ed61fcfd99b03e10dca65c3082d0728e|789303       |4929.0       |
|2025-07-01|fe5a8036be4b9a787b7c0ae013fc752a8cfb6c55a2f7b2fd152a6380925e9c49|789622       |3810.0       |
|2025-07-01|c68834ceaff15846bc1892c2185dc4e4f471d64fe3796b1a8ecc39a5a48c614f|789402       |960.0        |
+----------+----------------------------------------------------------------+-------------+-------------+
only showing top 5 rows


In [0]:
gold_parent_delta = DeltaTable.forName(spark, f"{catalog}.{gold_schema}.fact_orders")
gold_parent_delta.alias("parent_gold").merge(df_monthly.alias("child_gold"), "parent_gold.date = child_gold.date AND parent_gold.product_code = child_gold.product_code AND parent_gold.customer_code = child_gold.customer_code").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]